In [1]:
# === ColO-RAN: list all features (columns) by CSV type =======================
# Point this to your local scheduler folder (sched0 / sched1 / sched2)
from pathlib import Path
import os, re
import pandas as pd
from collections import defaultdict, Counter

In [3]:
# ---- Set this to your local scheduler folder (e.g., data/sched0) ------------
BASE_SCHED_DIR = Path(r"../data/sched0")   # raw string works on Windows; adjust if needed

# ---- Utilities ---------------------------------------------------------------
def find_csvs(root: Path):
    if not root.exists():
        print(f"[WARN] Path not found: {root.resolve()}")
        return []
    out = []
    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(".csv"):
                out.append(Path(r) / f)
    return out

def classify_file(path: Path):
    """
    Returns (entity_type, entity_id)
      - ue*.csv   -> ("ue", "ue#")
      - bs*.csv   -> ("bs", "bs#")
      - any CSV under a folder named like 'slices_bs*' -> ("slice", <filename_stem>)
      - else -> ("unknown", <filename_stem>)
    """
    name = path.name.lower()
    if re.fullmatch(r"ue\d+\.csv", name):
        return "ue", Path(name).stem
    if re.fullmatch(r"bs\d+\.csv", name):
        return "bs", Path(name).stem
    if any(seg.lower().startswith("slices_bs") for seg in path.parts):
        return "slice", path.stem
    return "unknown", path.stem

def parse_meta_from_parts(path: Path):
    parts = list(path.parts)
    get = lambda pat: next((s for s in parts if re.fullmatch(pat, s, flags=re.IGNORECASE)), None)
    return {
        "scheduler": get(r"sched\d+"),
        "tr":        get(r"tr\d+"),
        "exp":       get(r"exp\d+"),
        "bs":        get(r"bs\d+"),
    }

def build_catalog(root: Path) -> pd.DataFrame:
    rows = []
    for p in find_csvs(root):
        etype, eid = classify_file(p)
        meta = parse_meta_from_parts(p)
        rows.append({
            **meta,
            "entity_type": etype,
            "entity_id": eid,
            "path": str(p),
        })
    # Ensure expected columns exist even if rows is empty
    cols = ["scheduler","tr","exp","bs","entity_type","entity_id","path"]
    df = pd.DataFrame(rows)
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA
    # Safe sort only by columns that actually exist
    sort_keys = [c for c in ["entity_type","tr","exp","bs","entity_id","path"] if c in df.columns]
    if len(df) and sort_keys:
        df = df.sort_values(sort_keys).reset_index(drop=True)
    return df[cols]

def schema_union_and_intersection(file_paths, sample_rows=1000):
    """Return (union_df, intersection_list) for a list of CSV paths."""
    union_cols = set()
    inter_cols = None
    col_counts = Counter()
    dtype_sample = {}

    for fp in file_paths:
        try:
            df = pd.read_csv(fp, nrows=sample_rows)
        except Exception as e:
            print(f"[WARN] Could not read {fp}: {e}")
            continue
        cols = list(df.columns)
        union_cols.update(cols)
        inter_cols = set(cols) if inter_cols is None else (inter_cols & set(cols))
        for c in cols:
            col_counts[c] += 1
            if c not in dtype_sample:
                dtype_sample[c] = str(df[c].dtype)

    total = max(1, len(file_paths))
    rows = []
    for c in sorted(union_cols):
        rows.append({
            "column": c,
            "present_in_files": col_counts.get(c, 0),
            "presence_ratio": round(col_counts.get(c, 0) / total, 3),
            "example_dtype": dtype_sample.get(c, "—")
        })
    union_df = pd.DataFrame(rows).sort_values(
        ["presence_ratio","column"], ascending=[False, True]
    ).reset_index(drop=True)
    inter_list = sorted(list(inter_cols)) if inter_cols is not None else []
    return union_df, inter_list

# ---- Build catalog -----------------------------------------------------------
catalog = build_catalog(BASE_SCHED_DIR)
print(f"Found {len(catalog)} CSV files under {BASE_SCHED_DIR}")
display(catalog.head(20))

# ---- Compute feature inventories per type -----------------------------------
types_found = sorted(catalog["entity_type"].dropna().unique().tolist())
if not types_found:
    print("No CSV files found. Check BASE_SCHED_DIR path.")
else:
    for etype in ["ue", "slice", "bs", "unknown"]:
        subset = catalog[catalog["entity_type"] == etype]
        if subset.empty:
            continue
        print(f"\n=== {etype.upper()} files: {len(subset)} ===")
        paths = subset["path"].tolist()
        union_df, inter_cols = schema_union_and_intersection(paths)

        print(f"- Distinct columns (union): {len(union_df)}")
        print(f"- Columns present in ALL {etype.upper()} files ({len(inter_cols)}):")
        print(inter_cols[:50], "..." if len(inter_cols) > 50 else "")

        # Show top 50 columns by presence
        display(union_df.head(50))

        # Optional: save to CSV next to your sched folder
        out = BASE_SCHED_DIR / f"_features_{etype}.csv"
        try:
            union_df.to_csv(out, index=False)
            print(f"Saved: {out}")
        except Exception as e:
            print(f"[WARN] Could not save {out}: {e}")

# Tip: you can now open _features_ue.csv / _features_slice.csv / _features_bs.csv
# to see the full column lists. Next step: choose KPIs to plot.

Found 12973 CSV files under ../data/sched0


,scheduler,tr,exp,bs,entity_type,entity_id,path
0,sched0,tr0,exp1,bs1,bs,bs1,../data/sched0/tr0/exp1/bs1/bs1.csv
1,sched0,tr0,exp1,bs2,bs,bs2,../data/sched0/tr0/exp1/bs2/bs2.csv
2,sched0,tr0,exp1,bs3,bs,bs3,../data/sched0/tr0/exp1/bs3/bs3.csv
3,sched0,tr0,exp1,bs4,bs,bs4,../data/sched0/tr0/exp1/bs4/bs4.csv
4,sched0,tr0,exp1,bs5,bs,bs5,../data/sched0/tr0/exp1/bs5/bs5.csv
5,sched0,tr0,exp1,bs6,bs,bs6,../data/sched0/tr0/exp1/bs6/bs6.csv
6,sched0,tr0,exp1,bs7,bs,bs7,../data/sched0/tr0/exp1/bs7/bs7.csv
7,sched0,tr0,exp2,bs1,bs,bs1,../data/sched0/tr0/exp2/bs1/bs1.csv
8,sched0,tr0,exp2,bs2,bs,bs2,../data/sched0/tr0/exp2/bs2/bs2.csv
9,sched0,tr0,exp2,bs3,bs,bs3,../data/sched0/tr0/exp2/bs3/bs3.csv



=== UE files: 6044 ===
- Distinct columns (union): 21
- Columns present in ALL UE files (21):
['cc', 'cfo', 'dl_bler', 'dl_brate', 'dl_mcs', 'dl_snr', 'dl_turbo', 'earfcn', 'is_attached', 'pci', 'pl', 'rf_l', 'rf_o', 'rf_u', 'rsrp', 'time', 'ul_bler', 'ul_brate', 'ul_buff', 'ul_mcs', 'ul_ta'] 


,column,present_in_files,presence_ratio,example_dtype
0,cc,6044,1.0,int64
1,cfo,6044,1.0,float64
2,dl_bler,6044,1.0,float64
3,dl_brate,6044,1.0,float64
4,dl_mcs,6044,1.0,float64
5,dl_snr,6044,1.0,float64
6,dl_turbo,6044,1.0,float64
7,earfcn,6044,1.0,int64
8,is_attached,6044,1.0,float64
9,pci,6044,1.0,int64


Saved: ../data/sched0/_features_ue.csv

=== SLICE files: 5921 ===
- Distinct columns (union): 36
- Columns present in ALL SLICE files (36):
['IMSI', 'RNTI', 'Timestamp', 'Unnamed: 10', 'Unnamed: 18', 'Unnamed: 28', 'Unnamed: 31', 'Unnamed: 4', 'dl_buffer [bytes]', 'dl_cqi', 'dl_mcs', 'dl_n_samples', 'dl_pmi', 'dl_ri', 'num_ues', 'phr', 'power_multiplier', 'rx_brate uplink [Mbps]', 'rx_errors uplink (%)', 'rx_pkts uplink', 'scheduling_policy', 'slice_id', 'slice_prb', 'slicing_enabled', 'sum_granted_prbs', 'sum_requested_prbs', 'tx_brate downlink [Mbps]', 'tx_errors downlink (%)', 'tx_pkts downlink', 'ul_buffer [bytes]', 'ul_mcs', 'ul_n', 'ul_n_samples', 'ul_rssi', 'ul_sinr', 'ul_turbo_iters'] 


,column,present_in_files,presence_ratio,example_dtype
0,IMSI,5921,1.0,int64
1,RNTI,5921,1.0,int64
2,Timestamp,5921,1.0,int64
3,Unnamed: 10,5921,1.0,float64
4,Unnamed: 18,5921,1.0,float64
5,Unnamed: 28,5921,1.0,float64
6,Unnamed: 31,5921,1.0,float64
7,Unnamed: 4,5921,1.0,float64
8,dl_buffer [bytes],5921,1.0,int64
9,dl_cqi,5921,1.0,float64


Saved: ../data/sched0/_features_slice.csv

=== BS files: 1008 ===
- Distinct columns (union): 4
- Columns present in ALL BS files (4):
['dl_brate', 'nof_ue', 'time', 'ul_brate'] 


,column,present_in_files,presence_ratio,example_dtype
0,dl_brate,1008,1.0,float64
1,nof_ue,1008,1.0,int64
2,time,1008,1.0,int64
3,ul_brate,1008,1.0,float64


Saved: ../data/sched0/_features_bs.csv
